In [1]:
import ast
import numpy as np
import pandas as pd

<div dir="rtl"> 
שלב 1: בניית פונקציה prepare_date()

In [2]:
#עמודות Data Leakage
LEAKAGE_COLUMNS = ["averageRating", "numVotes", "BoxOffice"]

In [3]:
#בונים פונקציה שסופרת כמות שחקנים בסרט
def count_actors(actors_raw):
    if pd.isna(actors_raw):
        return 0
    try:
        actors_list = ast.literal_eval(actors_raw)
        return len(actors_list)
    except (ValueError, SyntaxError):
        # במידה והערך בטבלה לא רשימה
        return 0

In [11]:
def clean_genres(genres_raw):
    # ניקיון של סוגריים: ['Drama'] -> Drama. 
    #גם טיפול בערכים שלא עומדים בתנאי של LIST
    if pd.isna(genres_raw):
        return ""
    cleaned = genres_raw.replace("[", "").replace("]", "")
    cleaned = cleaned.replace("'", "").replace('"', "")
    parts = [part.strip() for part in cleaned.split(",")]
    return ",".join(parts)

In [12]:
def prepare_data(df):
    movies_df = df.copy()

    # טיפול בג'אנרים
    movies_df["genres"] = movies_df["genres"].apply(clean_genres)

    # startYear = 0    זה טעות, 0 יגרום למודל לטעות.
    movies_df["startYear"] = movies_df["startYear"].replace(0, np.nan)
    
    #פיטצר חדש: חישוב - כמות גאנרים בכל סרט וסרט
    movies_df["num_genres"] = movies_df["genres"].apply(
        lambda g: len(g.split(",")) if g else 0
    )

    # פיטצר חדש: חישוב - גאנר ראשי
    movies_df["main_genre"] = movies_df["genres"].apply(
        lambda g: g.split(",")[0] if g else "Unknown"
    )

    # פיטצר חדש: חישוב - באקרטינג לפי משך הסרט
    runtime_bins   = [0, 70, 85, 100, 120, 150, 400]
    runtime_labels = [1, 2, 3, 4, 5, 6]
    
    movies_df["runtime_bin"] = pd.cut(
        movies_df["runtimeMinutes"], bins=runtime_bins, labels=runtime_labels
    ).astype("float")

    #פיטצר חדש: יחס משך הסרט / מספר גאנרים בסרט   
    movies_df["runtime_per_genre"] = np.where(
        movies_df["num_genres"] > 0,
        movies_df["runtimeMinutes"] / movies_df["num_genres"],
        np.nan
    )
    #פיטצר חדש: משתמשים בפונקציה שבנינו מקודם ומחשבים כמה שחקנים בכל סרט
    movies_df["num_actors"] = movies_df["lead_actors_ids"].apply(count_actors)
    #פיטצר חדש (בינארי): האם בכלל קיימים שחקנים בסרט
    movies_df["is_no_cast"] = (movies_df["num_actors"] == 0).astype(int)


    #פיטצר חדש: כמה מילים יש בכל סרט
    movies_df["title_word_count"] = (
        movies_df["primaryTitle"].fillna("").apply(lambda t: len(t.split()))
    )

    #פיטצר חדש: האם יש נקודתיים בשם הסרט
    movies_df["title_has_colon"] = (
        movies_df["primaryTitle"].fillna("").str.contains(":").astype(int)
    )
    #פיטצר חדש: האם שפה של סרט הוא אנגלית
    movies_df["is_english"] = (movies_df["Language"] == "English").astype(int)
    #פיטצר חדש - האם סרט נוצר בארצות הברית
    movies_df["is_us"]      = (movies_df["Country"] == "United States").astype(int)

    #פיטצר חדש: האם קיים תקציב עבור כל סרט
    movies_df["has_budget"] = movies_df["budget"].notna().astype(int)

    feature_columns = [
        "startYear", "runtimeMinutes",
        "num_genres", "main_genre",
        "runtime_bin", "runtime_per_genre",
        "num_actors", "is_no_cast",
        "title_word_count", "title_has_colon",
        "is_english", "is_us",
        "has_budget",
    ]
    return movies_df[feature_columns]

<div dir="rtl"> 
בדיקה: האם פונקציה שבנינו עובדת

In [13]:
df = pd.read_csv("dataset.csv", low_memory=False)
data_prepared = prepare_data(df)

In [14]:
data_prepared.sample(5)

,startYear,runtimeMinutes,num_genres,main_genre,runtime_bin,runtime_per_genre,num_actors,is_no_cast,title_word_count,title_has_colon,is_english,is_us,has_budget
2462,1952.0,103.0,1,Comedy,4.0,103.0,5,0,2,0,0,0,0
71492,2011.0,80.0,1,Drama,2.0,80.0,5,0,3,0,0,0,0
6393,1939.0,75.0,0,Unknown,2.0,NaN,5,0,4,0,0,0,0
88014,1932.0,77.0,0,Unknown,2.0,NaN,5,0,3,0,0,0,0
54975,2018.0,132.0,2,Action,5.0,66.0,4,0,6,0,0,0,0


<div dir="rtl"> 
שלב 2: בניית מודל

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    GridSearchCV, KFold, cross_validate, cross_val_predict
)
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [17]:
#מוחקים מדאטה שהולך לאמן את המודל ערכים חסרים של משתנה מטרה
df_train = df.dropna(subset=["averageRating"]).copy()
df_train = df_train.reset_index(drop=True)

In [18]:
#בונים X ו Y, דאטה של פיטצרים ועמודה של ערך המטרה
X = prepare_data(df_train)
y = df_train["averageRating"].values

In [20]:
#נוודא גבולות של ערך המטרה
print(f"גודל של דאטה שהולך לאמן את המודל: {X.shape}")
print(f"ערך המטרה — min: {y.min():.1f}, max: {y.max():.1f}, ממוצע: {y.mean():.2f}")

גודל של דאטה שהולך לאמן את המודל: (115560, 13)
ערך המטרה — min: 1.0, max: 10.0, ממוצע: 6.07


In [23]:
#נבדוק כמה ערכים חסרים יש בכל פיטצר
missing = X.isnull().sum()
print("\nערכים חסרים בכל פיטצר:")
print(missing[missing > 0])


ערכים חסרים בכל פיטצר:
startYear               1
runtime_per_genre    1260
dtype: int64


In [24]:
# נחלק פיטצרים לפי קבוצות שלהם וכל קבוצה תקבל טרנספורם שונה
numeric_cols = [
    "startYear", "runtimeMinutes",
    "num_genres", "runtime_bin", "runtime_per_genre",
    "num_actors", "title_word_count"
]
binary_cols      = ["is_no_cast", "title_has_colon", "is_english", "is_us", "has_budget"]
categorical_cols = ["main_genre"]


In [25]:
#פיטצרים נומרים - לערכים חסרים נכניס חציון (לא רגיש לחריגים) וגם ננרמל נתונים
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

In [26]:
# פיטצרים בינרים - ערכים חסרים נמלא בערך שכיח
binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

In [27]:
#בפיצטרים קטגוריאלים נכניס במקום ערכים חסרים - ערך שכיח. בנוסף נבצע OneHotEncoder
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

In [29]:
#נבצע לכל סוג של פיטצר טיפול שונה ובסוף נקבל חזרה כדאטה מאוחד
preprocessor = ColumnTransformer(transformers=[
    ("numeric",     numeric_transformer,     numeric_cols),
    ("binary",      binary_transformer,      binary_cols),
    ("categorical", categorical_transformer, categorical_cols),
])

In [31]:
preprocessor

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['startYear', 'runtimeMinutes', 'num_genres',
                                  'runtime_bin', 'runtime_per_genre',
                                  'num_actors', 'title_word_count']),
                                ('binary',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['is_no_cast', 'title_has_colon', 'is_english',
                                  'is_us', 'has_budget']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['main_genre'])])